In [1]:
# Dependencies:
# pip install numpy pandas scikit-learn scipy matplotlib seaborn joblib

import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from urllib.parse import unquote

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn import svm
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
    roc_curve,
    auc,
)

# Configuration

In [2]:
DATASET_DIR        = Path("./Datasets")
ALL_DATASETS       = [1, 2, 3, 4]
OUTPUT_DIR         = Path("./outputs")

TFIDF_MAX_FEATURES = 5_000
TFIDF_NGRAM_RANGE  = (3, 5)
N_COMPONENTS       = 50
C_VALUES           = [0.0001, 0.01, 0.1, 10.0]
RANDOM_STATE       = 35
TRAIN_FRAC         = 0.70
VAL_FRAC           = 0.15

SECURITY_HEADERS = [
    "Cookie",
    "X-Forwarded-For",
    "Referer",
    "User-Agent",
    "Authorization",
    "X-Api-Key",
]

COLORS = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

# Data Processing Helpers

In [3]:
def loadAndFlatten(path):
    raw = pd.read_json(path)
    df  = pd.json_normalize(raw["request"])
    if "Attack_Tag" in df.columns:
        df["label"]       = df["Attack_Tag"].apply(
            lambda x: "Malicious" if pd.notna(x) and str(x).strip() != "" else "Benign"
        )
        df["attack_type"] = df["Attack_Tag"].fillna("Benign")
    else:
        df["label"]       = "Benign"
        df["attack_type"] = "Benign"
    df["_headers_raw"] = raw["request"].apply(
        lambda r: r.get("headers", {}) if isinstance(r, dict) else {}
    )
    return df

def buildText(row):
    parts = []

    for col in ["method", "url", "body"]:
        val = row.get(col, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val))

    headers = row.get("_headers_raw", {})
    if isinstance(headers, dict):
        for hname in SECURITY_HEADERS:
            hval = headers.get(hname, "")
            if hval and str(hval).strip():
                parts.append(f"{hname.lower()} {str(hval)}")

    text = " ".join(parts)
    try:
        text = unquote(text)
    except Exception:
        pass
    return text.lower().strip()

# Plots

In [4]:
def plotConfusionMatrices(allResults, modelType):
    n   = len(allResults)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()

    for i, result in enumerate(allResults):
        disp = ConfusionMatrixDisplay(confusion_matrix=result["test_cm"],
                                      display_labels=result["label_names"])
        disp.plot(cmap="Blues", ax=axes[i], colorbar=False)
        axes[i].set_title(f"Dataset {result["dataset"]}", fontsize=12)

    for j in range(n, 4):
        axes[j].set_visible(False)

    fig.suptitle(f"Confusion Matrices — {modelType}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / modelType / "cm_test.png", dpi=120)
    plt.close()


def plotMetrics(allResults, modelType):
    datasets    = [f"DS {r['dataset']}" for r in allResults]
    metrics     = ["accuracy", "precision", "recall", "f1"]
    metricNames = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
    x     = np.arange(len(datasets))
    width = 0.5

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for i, (metric, name) in enumerate(zip(metrics, metricNames)):
        vals = [r["test_metrics"][metric] for r in allResults]
        bars = axes[i].bar(x, vals, width, color=COLORS[:len(vals)], alpha=0.85)

        for bar, v in zip(bars, vals):
            axes[i].text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                         f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

        axes[i].set_title(name)
        axes[i].set_xticks(x)
        axes[i].set_xticklabels(datasets)
        axes[i].set_ylim(max(0, min(vals) - 0.05), 1.05)
        axes[i].set_ylabel("Score")
        axes[i].grid(axis="y", alpha=0.3)

    fig.suptitle(f"{modelType} — Test Metrics Across All Datasets",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / modelType / "metrics.png", dpi=120, bbox_inches="tight")
    plt.close()


def plotROCCurves(allResults, modelType):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for colIdx, r in enumerate(allResults):
        ax = axes[colIdx]
        fpr, tpr = r["roc"]
        aucVal   = r["auc"]

        ax.plot(fpr, tpr, color="darkorange", linewidth=2,
                label=f"AUC = {aucVal:.4f}")
        ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
        ax.fill_between(fpr, tpr, alpha=0.12, color="darkorange")
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1.02])
        ax.set_xlabel("False Positive Rate", fontsize=9)
        ax.set_ylabel("True Positive Rate", fontsize=9)
        ax.set_title(f"Dataset {r['dataset']}", fontsize=10)
        ax.legend(loc="lower right", fontsize=10)
        ax.grid(alpha=0.3)

    fig.suptitle(f"ROC Curves - {modelType}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / modelType / "roc_curves.png", dpi=120, bbox_inches="tight")
    plt.close()

# Data Pipeline and Training

In [5]:
def fitOneC(C, XTrain, yTrain, XVal, yVal):
    model = svm.LinearSVC(C=C, class_weight="balanced",
                          max_iter=5000, random_state=RANDOM_STATE)
    model.fit(XTrain, yTrain)
    yVPred = model.predict(XVal)
    prec, rec, f1, _ = precision_recall_fscore_support(
        yVal, yVPred, average="macro", zero_division=0)
    acc = accuracy_score(yVal, yVPred)
    return {"C": C, "val_f1": f1, "val_acc": acc,
            "val_prec": prec, "val_rec": rec}


def tuneAndEval(XTrain, yTrain, XVal, yVal, XTest, yTest):
    tuningRows = joblib.Parallel(n_jobs=len(C_VALUES), prefer="threads")(
        joblib.delayed(fitOneC)(C, XTrain, yTrain, XVal, yVal)
        for C in C_VALUES
    )
    tuningDf = pd.DataFrame(tuningRows)
    bestC    = tuningDf.loc[tuningDf["val_f1"].idxmax(), "C"]

    valModel = svm.LinearSVC(C=bestC, class_weight="balanced",
                             max_iter=5000, random_state=RANDOM_STATE)
    valModel.fit(XTrain, yTrain)
    yVPred = valModel.predict(XVal)
    vAcc = accuracy_score(yVal, yVPred)
    vp, vr, vf, _ = precision_recall_fscore_support(
        yVal, yVPred, average="macro", zero_division=0)
    valMetrics = {"accuracy": vAcc, "precision": vp, "recall": vr, "f1": vf}
    valCm = confusion_matrix(yVal, yVPred)

    XFinal = np.vstack([XTrain, XVal])
    yFinal = np.concatenate([yTrain, yVal])

    finalModel = svm.LinearSVC(C=bestC, class_weight="balanced",
                               max_iter=5000, random_state=RANDOM_STATE)
    t0 = time.time()
    finalModel.fit(XFinal, yFinal)
    trainTime = time.time() - t0

    t0     = time.time()
    yTPred = finalModel.predict(XTest)
    infTime = time.time() - t0
    tAcc = accuracy_score(yTest, yTPred)
    tp, tr, tf, _ = precision_recall_fscore_support(
        yTest, yTPred, average="macro", zero_division=0)
    testMetrics = {"accuracy": tAcc, "precision": tp, "recall": tr, "f1": tf,
                   "train_time": trainTime, "inference_time": infTime}
    testCm = confusion_matrix(yTest, yTPred)

    scores      = finalModel.decision_function(XTest)
    fpr, tpr, _ = roc_curve(yTest, scores)
    aucVal      = auc(fpr, tpr)

    return (tuningDf, bestC, valMetrics, valCm,
            testMetrics, testCm, (fpr, tpr), aucVal, finalModel)


def runDataset(datasetNum):
    trainPath = DATASET_DIR / f"dataset_{datasetNum}_train.json"

    df     = loadAndFlatten(trainPath)
    texts  = df.apply(buildText, axis=1)
    le     = LabelEncoder()
    labels = le.fit_transform(df["label"])
    labelNames = [str(c) for c in le.classes_]

    XTrTxt, XTmpTxt, yTr, yTmp = train_test_split(
        texts, labels,
        test_size=(VAL_FRAC + (1 - TRAIN_FRAC - VAL_FRAC)),
        random_state=RANDOM_STATE, stratify=labels,
    )
    XVTxt, XTeTxt, yVal, yTest = train_test_split(
        XTmpTxt, yTmp,
        test_size=0.5, random_state=RANDOM_STATE, stratify=yTmp,
    )

    tfidf = TfidfVectorizer(analyzer="char_wb",
                            ngram_range=TFIDF_NGRAM_RANGE,
                            max_features=TFIDF_MAX_FEATURES,
                            sublinear_tf=True, min_df=2)
    XTrRaw = tfidf.fit_transform(XTrTxt)
    XVRaw  = tfidf.transform(XVTxt)
    XTeRaw = tfidf.transform(XTeTxt)

    svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    svd.fit(XTrRaw)

    XTrPca = normalize(svd.transform(XTrRaw))
    XVPca  = normalize(svd.transform(XVRaw))
    XTePca = normalize(svd.transform(XTeRaw))

    (pcaTuning, pcaBestC,
     pcaValMet, pcaValCm,
     pcaTestMet, pcaTestCm,
     pcaRoc, pcaAuc,
     pcaModel) = tuneAndEval(XTrPca, yTr, XVPca, yVal, XTePca, yTest)

    joblib.dump({
        "tfidf":         tfidf,
        "svd":           svd,
        "pca_model":     pcaModel,
        "label_encoder": le,
        "config": {"dataset_num": datasetNum,
                   "best_C":      pcaBestC,
                   "n_components": N_COMPONENTS},
        "metrics": {"validation": pcaValMet, "test": pcaTestMet},
    }, OUTPUT_DIR / f"svm_pca_dataset{datasetNum}.joblib")

    labelCounts = df["label"].value_counts()
    return {
        "dataset":     datasetNum,
        "df":          df,
        "svd":         svd,
        "label_names": labelNames,
        "n_benign":    int(labelCounts.get("Benign",    0)),
        "n_malicious": int(labelCounts.get("Malicious", 0)),
        "tuning": {"pca": pcaTuning},
        "best_C":       pcaBestC,
        "val_metrics":  pcaValMet,
        "val_cm":       pcaValCm,
        "test_metrics": pcaTestMet,
        "test_cm":      pcaTestCm,
        "roc":          pcaRoc,
        "auc":          pcaAuc,
    }

# Prediction Helper For Pretrained Models

In [6]:
def predict(requests, artifactPath):
    art   = joblib.load(artifactPath)
    tfidf = art["tfidf"]
    le    = art["label_encoder"]
    X     = normalize(art["svd"].transform(tfidf.transform(pd.Series(requests))))
    enc   = art["pca_model"].predict(X)
    return le.inverse_transform(enc).tolist()

# Main

In [ ]:
def main():
    np.random.seed(RANDOM_STATE)
    sns.set_style("whitegrid")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    availableNums = [n for n in ALL_DATASETS
                     if (DATASET_DIR / f"dataset_{n}_train.json").exists()]
    for n in set(ALL_DATASETS) - set(availableNums):
        print(f"[SKIP] Dataset {n} not found: {DATASET_DIR / f'dataset_{n}_train.json'}")

    print(f"Running {len(availableNums)} dataset(s) in parallel...")
    allResults = joblib.Parallel(n_jobs=len(availableNums))(
        joblib.delayed(runDataset)(n) for n in availableNums
    )
    allResults = [r for r in allResults if r is not None]

    if not allResults:
        print("No datasets found. Check that ./Datasets/ folder exists.")
        return

    print("\nGenerating plots...")

    plotConfusionMatrices(allResults, "SVM")
    plotROCCurves(allResults, "SVM")
    plotMetrics(allResults, "SVM")

    print("\nResults Summary:")
    hdr = f"  {'Dataset':<12} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7}  {'Train(s)':>9}  {'Inf(ms)':>8}  Best C"
    print(hdr)
    print(f"  {'-' * (len(hdr) - 2)}")
    for r in allResults:
        m    = r["test_metrics"]
        aucV = r["auc"]
        print(f"  Dataset {r['dataset']:<4}  "
              f"{m['accuracy']:>7.4f} {m['precision']:>7.4f} "
              f"{m['recall']:>7.4f} {m['f1']:>7.4f} {aucV:>7.4f}  "
              f"{m['train_time']:>9.3f}  {m['inference_time']*1000:>8.2f}  "
              f"{r['best_C']}")

    if len(allResults) >= 2:
        f1s  = [r["test_metrics"]["f1"] for r in allResults]
        aucs = [r["auc"]                for r in allResults]
        print(f"\n  Avg Test F1 = {np.mean(f1s):.4f}  "
              f"(std {np.std(f1s):.4f})  |  Avg AUC = {np.mean(aucs):.4f}")

    print(f"\nAll outputs saved to: {OUTPUT_DIR.resolve()}")

main()

# Demo

In [7]:
# Dependencies:
# pip install fastapi uvicorn

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import uvicorn

app = FastAPI()

@app.middleware("http")
async def middleware(request: Request, call_next):
    body = await request.body()
    requestDict = {
        "method": request.method,
        "url": str(request.url),
        "body": body.decode(),
        "_headers_raw": dict(request.headers)
    }

    textInput = buildText(requestDict)
    predictions = predict([textInput], "./outputs/svm_pca_dataset1.joblib")

    if predictions[0] == "Malicious":
        return JSONResponse(
        status_code=403,
        content={
            "message": "Malicious Request"
        }
    )

    return await call_next(request)

@app.get("/api/data")
async def normalRoute():
    return JSONResponse(
        status_code=200,
        content={
            "message": "Benign Request"
        }
    )

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [1485]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52699 - "GET /api/data HTTP/1.1" 200 OK
INFO:     127.0.0.1:52709 - "GET /api/data?id=SELECT%20*%20FROM%20users%20WHERE%20admin%20=%20'true' HTTP/1.1" 403 Forbidden


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1485]
